# Colab gpu kernel playground

This notebook writes a tiny CUDA program, compiles it, and runs it on the Colab GPU.


In [41]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void vecAdd(float* A, float* B, float* C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    const int n = 1000000;
    float* h_A = new float[n];
    float* h_B = new float[n];
    float* h_C = new float[n];

    for (int i = 0; i < n; i++) {
        h_A[i] = 2 * i + 1;
        h_B[i] = 2 * i + 2;
    }

    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, n * sizeof(float));
    cudaMalloc((void**)&d_B, n * sizeof(float));
    cudaMalloc((void**)&d_C, n * sizeof(float));

    cudaMemcpy(d_A, h_A, n * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, n * sizeof(float), cudaMemcpyHostToDevice);

    cudaDeviceProp prop;
    cudaGetDeviceProperties(&prop, 0);

    int threadsPerBlock = prop.maxThreadsPerBlock;
    if (threadsPerBlock > n) {
        threadsPerBlock = n;
    }
    int blocks = (n + threadsPerBlock - 1) / threadsPerBlock;

    std::cout << "Max threads per block: " << prop.maxThreadsPerBlock << std::endl;
    std::cout << "Launching " << blocks << " blocks of " << threadsPerBlock << " threads" << std::endl;

    vecAdd<<<blocks, threadsPerBlock>>>(d_A, d_B, d_C, n);

    cudaDeviceSynchronize();

    cudaMemcpy(h_C, d_C, n * sizeof(float), cudaMemcpyDeviceToHost);

    // for (int i = 0; i < n; i++) {
    //     std::cout << h_A[i] << " + " << h_B[i] << " = " << h_C[i] << std::endl;
    // }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Overwriting vector_add.cu


In [31]:
!nvidia-smi

Tue Apr 28 20:53:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [42]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [43]:
!./vector_add

Max threads per block: 1024
Launching 977 blocks of 1024 threads


In [44]:
!ncu ./vector_add

==PROF== Connected to process 14485 (/content/vector_add)
Max threads per block: 1024
Launching 977 blocks of 1024 threads
==PROF== Profiling "vecAdd" - 0: 0%....50%....100% - 9 passes
==PROF== Disconnected from process 14485
[14485] vector_add@127.0.0.1
  vecAdd(float *, float *, float *, int) (977, 1, 1)x(1024, 1, 1), Context 1, Stream 7, Device 0, CC 7.5
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    DRAM Frequency                  Ghz         4.74
    SM Frequency                    Mhz       584.26
    Elapsed Cycles                cycle       29,939
    Memory Throughput                 %        83.45
    DRAM Throughput                   %        83.45
    Duration                         us        51.23
    L1/TEX Cache Throughput           %        27.64
    L2 Cache Throughput               %        26.90
    SM Active 